# F1_p2 — Track F1/F2, repeated random patient-wise 4-fold CV (FUTURE_RESEARCH_PLAN.md rev 1)

**Runs:** partition 2 (seed 2): GAE + MLP autoencoder, 4 folds each (8 jobs), 200 epochs. **Account:** account 2 (nhannguyen26). **Accelerator:** None (CPU) *or* GPU T4 x2 — the code uses whichever is set
(chosen from the smoke measurement). **Internet:** on. **Inputs:** public datasets `v9-x3-fold1` + `v9-x3-fold3`
(all 24 CHB-MIT cases); to resume an interrupted run, also add this notebook's previous version output.
Code: `github.com/Nhanvas/Thesis` at `1e653fa` (read only) + `f1_core.py` (sha256 `bf4bfd69d7669226`), `f1_run.py`
(`8768801adf6b6cfb`) and `partitions.json` (`fbf73129105e0fea`) below.
Order: setup → D0 → gates (G-const, G-vec, G-mlp, GG0) → smoke (2-epoch end-to-end + speed, projection) →
full run (each job written as soon as it ends) → `/kaggle/working/f1_repcv/p2/`.
**Nothing printed here is a result** until verification (`src/f1_aggregate.py`). Do not edit;
*Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository at the pinned commit (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
os.environ["F1_T0"] = str(T_NB)
NOTEBOOK, P, RUN_FULL = "F1_p2", 2, True
REPO_COMMIT = "1e653fa233344f262379bfc11b8c8b172fe0d562"
TEST = bool(os.environ.get("F1_TEST_REPO"))                      # synthetic test in the build container only
WORK = os.environ.get("F1_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("F1_TEST_INPUT", "/kaggle/input")
SRC = WORK + "/f1_src"
os.makedirs(SRC, exist_ok=True)
if TEST:
    REPO = os.environ["F1_TEST_REPO"]
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/p2_scalars.py"):
        r = subprocess.run(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/Nhanvas/Thesis.git "
                           f"&& git fetch -q --depth 1 origin {REPO_COMMIT} && git checkout -q FETCH_HEAD", shell=True)
        if r.returncode != 0:
            subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                                  f"&& cd {REPO} && git checkout -q {REPO_COMMIT}", shell=True)
    head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
    assert head == REPO_COMMIT, f"wrong commit {head}"
print("repository", REPO, f"({time.time()-T_NB:.0f}s)")

In [ ]:
%%writefile {SRC}/f1_core.py
#!/usr/bin/env python
"""
f1_core.py -- Track F1/F2 (docs/FUTURE_RESEARCH_PLAN.md rev 1): repeated random patient-wise 4-fold CV of the GAE and
of the MLP autoencoder (the GAE without message passing). Partition-aware generalisation of the closed model
comparison (src/model_cmp_core.py, imported read-only, never edited): every fold list, every normalisation constant
and every k comes from the partition being run (results/f1_repcv/partitions.json); nothing is tied to the committed
folds except partition 0, which is only used for gates.

Per (partition p, model, fold f) job:
  1. f7 normalisation constants from the fold's training cases (clean windows; the computation of
     v9_x3_train_fold.norm_constants_fold, float64 running sums, cases in sorted order);
  2. training: the thesis fold recipe (variant "clean" windows of the training cases, node input 30 = adjacency row /
     max 18 + band powers per-window min-max 5 + f7 standardised, clipped +-4, mapped to [0, 1] 7; Adam 1e-3, cosine
     T_max = 200, batch 32, 200 epochs, last epoch kept, no validation subset, no early stopping), seed = the
     partition's seed. GAE = gae_joint.GAEModel (GCNConv 30 -> 64 -> 16, decoders as the thesis); MLP-AE =
     model_cmp_core.build_mlp_ae (Linear 30 -> 64 -> 16, same decoders). Both 3,964 parameters. The batches are
     built with tensors instead of the PyG DataLoader; the window order reproduces DataLoader(shuffle=True,
     generator=seed) exactly (gate G-vec: identical weights to the PyG loop on CPU);
  3. the frozen downstream chain, unchanged (v9_e1_baselines.evaluate_arm via the model comparison's construction):
     per-channel LedoitWolf distance of the 16-d node latent -> members 6/9/12 + mean with +-15 min drift ->
     percentile mean (x) / robust-z member mean (m) -> smoothing 5 -> per-patient budget clip(q_i(k), 0.005, 0.02),
     k calibrated on the fold's own training cases (median q_i = 0.005, label-free) -> merge 90 s -> kernel CPD ->
     SzCORE; plus the fixed 0.005 point, the detector curve over v9_e1_baselines.GRID, window AUROC, per-seizure caught.

Gates (f1_run.py calls them; STOP on failure): D0 data identity (model_cmp_core.gate_D0); G-const (partition 0's
constants recomputed here = the committed fold models' constants); GG0 (this module's chain, run on partition 0 with
the committed GAE fold checkpoints, reproduces results/v9_x3 and the 185 caught flags exactly; k within 1e-6
relative); G-vec (the tensor training loop gives the same weights as the PyG DataLoader loop of v9_x3_train_fold.py,
CPU, real windows); G-mlp (this module's MLP-AE loop = model_cmp_core.mlp_train, CPU); per job: training-window count
= the committed clean-window counts of the training cases, parameter count 3,964, G2 (last-epoch loss < first).
"""
import hashlib
import json
import os
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path(os.environ.get("THESIS_REPO", Path(__file__).resolve().parents[1]))
for _p in ("src", "src/retrain", "src/phaseB", "src/phaseC", "src/dataprep"):
    if str(REPO / _p) not in sys.path:
        sys.path.insert(0, str(REPO / _p))

import model_cmp_core as C               # noqa: E402  (closed code, read only)

V9E1 = C.V9E1
PARTITIONS = Path(os.environ.get("F1_PARTITIONS", REPO / "results" / "f1_repcv" / "partitions.json"))
EPOCHS = int(os.environ.get("F1_EPOCHS", 200))       # 200 = the registered recipe; any other value is a test only
BATCH, LR, N_CH, IN_DIM, OUT_X, LAT = 32, 1e-3, 18, 30, 12, 16
MODELS = ("gae", "mlp")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def sha16(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]


# ============================================================================ partitions
_PARTS = None


def parts():
    global _PARTS
    if _PARTS is None:
        _PARTS = json.loads(PARTITIONS.read_text())["partitions"]
    return _PARTS


def manifest(p):
    """{fold (int): dict(held_out_cases, train_cases)} of partition p."""
    return {int(k): v for k, v in parts()[str(p)]["folds"].items()}


def seed_of(p):
    return int(parts()[str(p)]["seed"])


# ============================================================================ normalisation constants + window counts
def clean_counts():
    """clean training-window count of every case, read from the committed fold models' metas (each case is a
    training case of three of them); STOP if they disagree."""
    out = {}
    for f in C.FOLDS:
        for s, n in C.fold_meta(f)["n_train"].items():
            if s in out and out[s] != int(n):
                raise SystemExit(f"STOP: committed clean-window counts of {s} differ between fold models")
            out[s] = int(n)
    if sorted(out) != sorted(C.ALL):
        raise SystemExit(f"STOP: clean-window counts missing for {sorted(set(C.ALL) - set(out))}")
    return out


def fold_constants(train_cases):
    """v9_x3_train_fold.norm_constants_fold on the fold's training cases (sorted), clean windows, float64 sums."""
    n, s1, s2 = 0, np.zeros(7), np.zeros(7)
    for s in sorted(train_cases):
        z = C.npz(s)
        F = z["f7"][C.clean_mask(z)].reshape(-1, 7).astype(np.float64)
        if not np.isfinite(F).all():
            raise SystemExit(f"STOP: {s}: non-finite f7 inside the training mask")
        n += len(F); s1 += F.sum(0); s2 += (F * F).sum(0)
    mu = s1 / n
    return mu, np.sqrt(np.maximum(s2 / n - mu * mu, 1e-12))


def new_meta(p, f):
    man = manifest(p)[f]
    mu, sd = fold_constants(man["train_cases"])
    cc = clean_counts()
    return dict(partition=int(p), fold=int(f), seed=seed_of(p), in_dim=IN_DIM, out_x=OUT_X, clip=4.0,
                add_mean=mu.tolist(), add_std=sd.tolist(), train_cases=sorted(man["train_cases"]),
                held_out_cases=list(man["held_out_cases"]), n_train={s: cc[s] for s in sorted(man["train_cases"])})


# ============================================================================ training (tensor batches)
def _loss(kind, model, A, X, T):
    """train_gae_v2.batch_loss (kind 'gae') / model_cmp_core.mlp_loss (kind 'mlp') on tensor batches:
    MSE(A_raw, clamp(Z Z^T, 0, 1)) + 0.1 MSE([xn, fa], X_hat) per window."""
    import torch
    import gae_joint as G
    B = A.shape[0]
    if kind == "gae":
        b_, r_, c_ = A.nonzero(as_tuple=True)                        # = dense_to_sparse per window + Batch offsets
        ei = torch.stack([b_ * N_CH + r_, b_ * N_CH + c_])
        z = model.encoder(X.reshape(B * N_CH, IN_DIM), ei, A[b_, r_, c_])
    else:
        z = model.encoder(X.reshape(B * N_CH, IN_DIM))
    zp = z.view(B, N_CH, LAT)
    Ah = torch.clamp(torch.bmm(zp, zp.transpose(1, 2)), 0.0, 1.0)
    Xh = model.x_decoder(z).view(B, N_CH, OUT_X)
    return ((A - Ah) ** 2).mean(dim=(1, 2)) + G.LAMBDA * ((T - Xh) ** 2).mean(dim=(1, 2))


def build(kind):
    import gae_joint as G
    return G.GAEModel(in_dim=IN_DIM, bands=OUT_X) if kind == "gae" else C.build_mlp_ae()


def train(kind, cases, meta, seed, device, epochs=None, max_windows=None, tag=""):
    """the thesis fold recipe; window order = torch DataLoader(shuffle=True, generator seeded `seed`): per epoch one
    int64 draw (iterator base seed), one randperm (the order), one tail randperm at exhaustion."""
    import torch
    epochs = EPOCHS if epochs is None else epochs
    t_load = time.time()
    A, X, T = C._mlp_tensors(sorted(cases), meta, device, C.clean_mask, max_windows)
    n = len(A)
    t_load = time.time() - t_load
    torch.manual_seed(seed); np.random.seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    g = torch.Generator().manual_seed(seed)
    model = build(kind).to(device)
    n_params = sum(q.numel() for q in model.parameters())
    if n_params != 3964:
        raise SystemExit(f"STOP: {kind} parameter count {n_params} != 3964")
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    rows, t0 = [], time.time()
    for ep in range(epochs):
        model.train()
        tot = torch.zeros((), device=device)
        torch.empty((), dtype=torch.int64).random_(generator=g)
        perm = torch.randperm(n, generator=g).to(device)
        torch.randperm(n, generator=g)
        for i in range(0, n, BATCH):
            ii = perm[i:i + BATCH]
            opt.zero_grad()
            loss = _loss(kind, model, A[ii], X[ii], T[ii]).mean()
            loss.backward(); opt.step()
            tot += loss.detach() * len(ii)
        sched.step()
        rows.append(dict(model=kind, epoch=ep + 1, train_loss=float(tot) / n, s=round(time.time() - t0, 1)))
        if ep < 2 or (ep + 1) % 20 == 0 or ep == epochs - 1:
            log(f"[{kind}{tag}] epoch {ep+1}/{epochs} loss {rows[-1]['train_loss']:.6f} ({rows[-1]['s']}s)")
    steps = epochs * ((n + BATCH - 1) // BATCH)
    info = dict(model=kind, n_train=int(n), epochs=epochs, batch=BATCH, lr=LR, seed=int(seed), n_params=n_params,
                first_loss=rows[0]["train_loss"], last_loss=rows[-1]["train_loss"], fit_s=round(time.time() - t0, 1),
                load_s=round(t_load, 1), ms_per_step=round((time.time() - t0) / steps * 1000, 3), device=str(device),
                G2="PASS" if rows[-1]["train_loss"] < rows[0]["train_loss"] else "FAIL", torch=torch.__version__)
    return model.eval(), info, rows


# ============================================================================ per-channel distance D
def encode_D(kind, model, meta, ctx, chunk=2048):
    """node latent on every valid window (dump convention of v9_x3_cv_eval.dump_D / model_cmp_core.node_input), on
    CPU, then the per-channel LedoitWolf Mahalanobis distance. kind 'gae' = model_cmp_core.gae_D with an explicit
    model; kind 'mlp' = model_cmp_core.mlp_D on CPU."""
    import torch
    model = model.cpu().eval()
    X30, adj, xn, fa = C.node_input(ctx["z"], ctx["idx"], meta)
    Z = np.empty((len(adj), N_CH, LAT), np.float32)
    with torch.no_grad():
        for s in range(0, len(adj), chunk):
            if kind == "gae":
                A = torch.tensor(adj[s:s + chunk]); B = A.shape[0]
                An = A / (A.amax(dim=(1, 2), keepdim=True) + 1e-8)
                x = torch.cat([An, torch.tensor(xn[s:s + chunk]), torch.tensor(fa[s:s + chunk])], 2).reshape(B * N_CH, -1)
                b, i, j = A.nonzero(as_tuple=True)
                ei = torch.stack([b * N_CH + i, b * N_CH + j])
                Z[s:s + chunk] = model.encoder(x, ei, A[b, i, j]).view(B, N_CH, LAT).numpy()
            else:
                xb = torch.tensor(X30[s:s + chunk])
                Z[s:s + chunk] = model.encoder(xb.reshape(-1, IN_DIM)).view(-1, N_CH, LAT).numpy()
    return C.d_from_Z(Z)


# ============================================================================ chain + outputs
def seizure_rows(model, p, f, ctx, loc):
    rows = []
    for fi, rr in C.P3L.refs_of(ctx).items():
        for a, b in rr:
            hit = any(h1 >= a - C.PRE and h0 <= b + C.POST for h0, h1 in loc.get(fi, []))
            rows.append(dict(model=model, partition=p, subject=ctx["s"], fold=f,
                             file=ctx["files"][int(fi)] if ctx["files"] else int(fi), onset_s=int(a),
                             duration_s=int(b - a), caught=bool(hit)))
    return rows


def evaluate_case(model, p, f, ctx, x, m, k):
    """model_cmp_core.evaluate_case with the fold and partition given explicitly."""
    from sklearn.metrics import roc_auc_score
    per = []
    out, _det, loc = V9E1.evaluate_arm(ctx, x, k, C.FLOOR, C.CAP, C.SMOOTH, m=m)
    for st, v in out.items():
        per.append(dict(model=model, partition=p, fold=f, subject=ctx["s"], point="own", stage=st, k=k, **v))
    seiz = seizure_rows(model, p, f, ctx, loc)
    out0, _, _ = V9E1.evaluate_arm(ctx, x, k, C.FLOOR, C.CAP, C.SMOOTH, m=None)
    for st, v in out0.items():
        per.append(dict(model=model, partition=p, fold=f, subject=ctx["s"], point="fixed0.005", stage=st, k=np.nan, **v))
    cur = [dict(model=model, partition=p, fold=f, subject=ctx["s"], **r) for r in V9E1.sweep_detector(ctx, x, C.GRID)]
    ok = ctx["ok"]; y = ctx["ict"][ok].astype(int)
    au = dict(model=model, partition=p, fold=f, subject=ctx["s"],
              auroc=float(roc_auc_score(y, x[ok])) if 0 < y.sum() < len(y) else np.nan)
    return per, cur, [au], seiz


def run_fold(model, p, f, man, D_fn):
    """D_fn(ctx) -> per-channel distance. k from the fold's training cases; held-out cases evaluated."""
    t0 = time.time()
    ms = []
    for s in sorted(man["train_cases"]):
        ctx = C.make_ctx(s)
        ms.append((ctx, C.scalars("D", ctx, D_fn(ctx))[1]))
    k = C.calibrate(ms)
    del ms
    P, Cc, A, S = [], [], [], []
    for s in man["held_out_cases"]:
        ctx = C.make_ctx(s)
        x, m = C.scalars("D", ctx, D_fn(ctx))
        a, b, c, d = evaluate_case(model, p, f, ctx, x, m, k)
        P += a; Cc += b; A += c; S += d
    log(f"[{model}] partition {p} fold {f}: k = {k:.6f}; held-out {man['held_out_cases']} scored ({time.time()-t0:.0f}s)")
    return P, Cc, A, S, k, time.time() - t0


# ============================================================================ gates
def gate_const():
    """partition 0: constants and training-window counts computed here = the committed fold models' metas."""
    worst, bad = 0.0, []
    for f in C.FOLDS:
        ref = C.fold_meta(f)
        mu, sd = fold_constants(manifest(0)[f]["train_cases"])
        for got, want in ((mu, ref["add_mean"]), (sd, ref["add_std"])):
            r = float(np.max(np.abs(got - np.array(want)) / np.maximum(np.abs(want), 1e-12)))
            worst = max(worst, r)
            if r > 1e-9:
                bad.append((f, r))
        if sorted(ref["n_train"]) != sorted(manifest(0)[f]["train_cases"]):
            bad.append((f, "training cases differ from the committed fold model"))
    st = "PASS" if not bad else "FAIL"
    log(f"G-const {st}: partition-0 constants vs committed fold models, max relative difference {worst:.2e}")
    return dict(gate="G-const", status=st, max_rel=worst, bad=str(bad))


def committed_gae(f):
    import torch
    meta = C.fold_meta(f)
    m = build("gae")
    m.load_state_dict(torch.load(str(C.X3 / "models" / f"fold{f}" / f"gae_fold{f}_seed42.pt"), map_location="cpu"),
                      strict=True)
    return m.eval(), meta


def gate_GG0():
    """this module's chain on partition 0 with the committed GAE fold checkpoints = results/v9_x3 (tp/fp of 48 rows
    and the 185 caught flags exactly, k within 1e-6 relative; model_cmp_core.gate_GG)."""
    t0 = time.time()
    P, S, ks, secs = [], [], {}, []
    for f in C.FOLDS:
        model, meta = committed_gae(f)
        p_, _c, _a, s_, k, sec = run_fold("gae", 0, f, manifest(0)[f], lambda ctx: encode_D("gae", model, meta, ctx))
        P += p_; S += s_; ks[f] = k; secs.append(sec)
    g = C.gate_GG(P, S, ks)
    g.update(gate="GG0", seconds=round(time.time() - t0), fold_seconds=[round(x) for x in secs])
    return g


def _pyg_reference_train(cases, meta, seed, epochs, max_windows):
    """v9_x3_train_fold.py's loop verbatim (PyG Data objects + DataLoader), CPU, on the given windows."""
    import torch
    from torch_geometric.loader import DataLoader
    import train_gae_v2 as TGV2
    mu, sd = np.array(meta["add_mean"]), np.array(meta["add_std"])
    ds = []
    for s in sorted(cases):
        z = C.npz(s)
        idx = np.flatnonzero(TGV2.mask(z, "clean"))[:max_windows]
        adj = z["adj"][idx].astype(np.float32)
        xn = TGV2.bp_minmax(z["bp5"][idx].astype(np.float32))
        fa = TGV2.scale_add(z["f7"][idx].astype(np.float64), mu, sd)
        ds += [TGV2.window_data(adj[i], xn[i], fa[i]) for i in range(len(idx))]
    TGV2.set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = build("gae")
    tl = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for _ in range(epochs):
        model.train()
        for b in tl:
            opt.zero_grad()
            TGV2.batch_loss(model, b, torch.device("cpu")).mean().backward(); opt.step()
        sched.step()
    return model


def _maxdiff(m1, m2):
    a, b = m1.state_dict(), m2.state_dict()
    return float(max((a[k].cpu() - b[k].cpu()).abs().max().item() for k in a))


def gate_vec(max_windows=2000, epochs=2):
    """the tensor training loop = the PyG DataLoader loop of v9_x3_train_fold.py (CPU, partition-0 fold-1 first
    training case, committed fold-1 constants, seed 42)."""
    import torch
    t0 = time.time()
    case = sorted(manifest(0)[1]["train_cases"])[:1]
    meta = C.fold_meta(1)
    ref = _pyg_reference_train(case, meta, 42, epochs, max_windows)
    mine, _, _ = train("gae", case, meta, 42, torch.device("cpu"), epochs=epochs, max_windows=max_windows, tag=" G-vec")
    d = _maxdiff(ref, mine)
    st = "PASS" if d <= 1e-6 else "FAIL"
    log(f"G-vec {st}: tensor loop vs PyG DataLoader loop, max |weight difference| = {d:.3e} ({time.time()-t0:.0f}s)")
    return dict(gate="G-vec", status=st, max_abs_diff=d, exact=d == 0.0, case=case[0], windows=max_windows, epochs=epochs)


def gate_mlp(epochs=2):
    """this module's MLP-AE loop = model_cmp_core.mlp_train (the run of the model comparison), CPU, smoke subset."""
    import torch
    t0 = time.time()
    ref, _, _ = C.mlp_train(1, torch.device("cpu"), smoke=True)
    case = C.MAN["1"]["train_cases"][:1]
    mine, _, _ = train("mlp", case, C.fold_meta(1), 42, torch.device("cpu"), epochs=epochs, max_windows=2000, tag=" G-mlp")
    d = _maxdiff(ref, mine)
    st = "PASS" if d <= 1e-6 else "FAIL"
    log(f"G-mlp {st}: MLP-AE loop vs model_cmp_core.mlp_train, max |weight difference| = {d:.3e} ({time.time()-t0:.0f}s)")
    return dict(gate="G-mlp", status=st, max_abs_diff=d, exact=d == 0.0)


# ============================================================================ one job = (partition, model, fold)
def job(p, model, f, device, out_root, smoke=False):
    """train + score one fold; every output written under out_root/p{p}/jobs/{model}_fold{f}/ (smoke: .../smoke/),
    DONE.json written last (a job without DONE.json does not exist)."""
    import resource
    import torch
    t0 = time.time()
    jd = Path(out_root) / f"p{p}" / ("smoke" if smoke else "jobs") / f"{model}_fold{f}"
    jd.mkdir(parents=True, exist_ok=True)
    man = manifest(p)[f]
    meta = new_meta(p, f)
    seed = seed_of(p)
    cases = meta["train_cases"]
    mdl, info, rows = train(model, cases[:1] if smoke else cases, meta, seed, device,
                            epochs=2 if smoke else None, max_windows=2000 if smoke else None, tag=f" p{p} f{f}")
    if not smoke and info["n_train"] != sum(meta["n_train"].values()):
        raise SystemExit(f"STOP: {model} p{p} fold {f}: training windows {info['n_train']} != committed clean "
                         f"counts {sum(meta['n_train'].values())}")
    tag = f"{model}_p{p}_fold{f}_seed{seed}" + ("_SMOKE" if smoke else "")
    torch.save({k: v.cpu() for k, v in mdl.state_dict().items()}, jd / f"{tag}.pt")
    (jd / f"{tag}.json").write_text(json.dumps(dict(meta, model=model, **{k: v for k, v in info.items()
                                                                          if k != "model"}), indent=1))
    pd.DataFrame([dict(r, partition=p, fold=f) for r in rows]).to_csv(jd / "training_log.csv", index=False)
    if smoke:
        ctx = C.make_ctx(man["held_out_cases"][0])
        D = encode_D(model, mdl, meta, ctx)
        if not np.isfinite(D).all():
            raise SystemExit(f"STOP smoke: non-finite {model} distance on {ctx['s']}")
        cal = C.make_ctx(cases[0])
        k = C.calibrate([(cal, C.scalars("D", cal, encode_D(model, mdl, meta, cal))[1])])
        per, cur, au, seiz = evaluate_case(model, p, f, ctx, *C.scalars("D", ctx, D), k)
        res = (per, cur, au, seiz, k, 0.0)
    else:
        res = run_fold(model, p, f, man, lambda ctx: encode_D(model, mdl, meta, ctx))
    per, cur, au, seiz, k, score_s = res
    for name, rr in (("per_subject", per), ("curves", cur), ("auroc", au), ("seizures", seiz)):
        pd.DataFrame(rr).to_csv(jd / f"{name}.csv", index=False)
    fit = dict(info, partition=p, fold=f, k=k, score_s=round(score_s, 1), smoke=smoke,
               peak_rss_mb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
               core_sha=sha16(Path(__file__)), partitions_sha=sha16(PARTITIONS))
    (jd / "fit.json").write_text(json.dumps(fit, indent=1))
    if info["G2"] != "PASS":
        raise SystemExit(f"STOP G2: {model} p{p} fold {f} did not train (loss did not fall)")
    (jd / "DONE.json").write_text(json.dumps(dict(status="done", seconds=round(time.time() - t0), **{
        k: fit[k] for k in ("core_sha", "partitions_sha", "device", "epochs", "seed")}), indent=1))
    log(f"[{model}] p{p} fold {f} {'SMOKE ' if smoke else ''}done in {time.time()-t0:.0f}s "
        f"(train {info['fit_s']}s, {info['ms_per_step']} ms/step, peak RSS {fit['peak_rss_mb']} MB)")
    return dict(model=model, partition=p, fold=f, status="done", seconds=round(time.time() - t0),
                ms_per_step=info["ms_per_step"], device=str(device))


def timing(device, n_steps=300, n_windows=20000):
    """measured ms per training step (batch 32) of both models on real windows (fold-1 first training case)."""
    import torch
    meta = C.fold_meta(1)
    A, X, T = C._mlp_tensors(sorted(manifest(0)[1]["train_cases"])[:1], meta, device, C.clean_mask, n_windows)
    out = {}
    for kind in MODELS:
        model = build(kind).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        for rep in range(2):                                   # rep 0 = warm-up
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            t0 = time.time()
            for _ in range(n_steps):
                ii = torch.randint(0, len(A), (BATCH,), device=device)
                opt.zero_grad(); _loss(kind, model, A[ii], X[ii], T[ii]).mean().backward(); opt.step()
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            out[kind] = (time.time() - t0) / n_steps * 1000
    return out

In [ ]:
%%writefile {SRC}/f1_run.py
#!/usr/bin/env python
"""
f1_run.py -- driver of the Track F1/F2 Kaggle notebooks (docs/FUTURE_RESEARCH_PLAN.md rev 1). One notebook = one
partition p, both models (GAE, MLP-AE), 4 folds = 8 atomic jobs, run by W parallel worker processes:
  Accelerator None    -> W = min(4, CPU cores) workers on the CPU, 1 thread each (deterministic);
  Accelerator GPU T4x2 -> W = 4 workers, alternating cuda:0 / cuda:1.
Order: setup (D0) -> gates (G-const, GG0, G-vec, G-mlp) -> smoke (one tiny end-to-end job per model + timing of W
concurrent workers -> projection.json; the full run is refused if it cannot fit) -> full run -> merge.
Each job writes its own folder and DONE.json last; a job is started only if its projected end is before
SESSION_STOP_H. A previous version's output attached as input is reused job by job (same code and partitions sha only).
Nothing printed here is a result before verification (src/f1_aggregate.py).
"""
import json
import multiprocessing as mp
import os
import platform
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

import f1_core as F

OUT = Path(os.environ.get("F1_OUT", "/kaggle/working/f1_repcv"))
T0 = float(os.environ.get("F1_T0", time.time()))
os.environ["F1_T0"] = str(T0)        # spawned workers import this module with the parent's environment
SESSION_STOP_H = 11.3            # Kaggle stops a session at 12 h
REFUSE_H = 10.5                  # the full run is refused if its projected wall time exceeds this
GATES = []
PROJ = {}


def hours():
    return (time.time() - T0) / 3600


def _json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=str)); tmp.replace(path)


def gate(rec, p):
    GATES.append(rec); _json(OUT / f"p{p}" / "gates.json", GATES)
    if rec.get("status") != "PASS":
        raise SystemExit(f"STOP gate {rec.get('gate')}: {rec}")


# ============================================================================ devices
GB_PER_WORKER = float(os.environ.get("F1_GB_PER_WORKER", 6.0))   # host-RAM budget per worker (largest fold ~3.5 GB estimated, not measured; peak RSS is recorded per job)


def devices():
    """W = min(4, CPU cores, host RAM / 6 GB) workers; GPU mode alternates cuda:0 / cuda:1."""
    import torch
    mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    w = max(1, min(4, os.cpu_count() or 1, int(mem_gb // GB_PER_WORKER)))
    n_gpu = torch.cuda.device_count()
    if n_gpu:
        return [f"cuda:{i % n_gpu}" for i in range(w)], "gpu"
    return ["cpu"] * w, "cpu"


def _init(slots, env):
    os.environ.update(env)
    import sys
    for q in env.get("F1_SYS_PATH", "").split(os.pathsep):
        if q and q not in sys.path:
            sys.path.insert(0, q)
    import torch
    torch.set_num_threads(1)
    os.environ["F1_DEVICE"] = slots.get()


def _env():
    import sys
    return dict(THESIS_REPO=str(F.REPO), F1_PARTITIONS=str(F.PARTITIONS), F1_EPOCHS=str(F.EPOCHS), F1_OUT=str(OUT),
                F1_T0=str(T0), F1_SYS_PATH=os.pathsep.join(sys.path), F1_INPUT=os.environ.get("F1_INPUT", "/kaggle/input"))


def _worker_setup():
    import torch
    F.C.locate_all(os.environ["F1_INPUT"])
    return torch.device(os.environ["F1_DEVICE"])


def _timing_worker(_):
    dev = _worker_setup()
    return dict(device=str(dev), **F.timing(dev))


def _job_worker(spec):
    p, model, f, proj_h = spec
    dev = _worker_setup()
    if hours() + proj_h * 1.15 > SESSION_STOP_H:
        return dict(model=model, partition=p, fold=f, status="NOT RUN (session time)", projected_h=proj_h)
    try:
        return F.job(p, model, f, dev, OUT)
    except BaseException as e:                                            # noqa: BLE001
        import traceback
        jd = OUT / f"p{p}" / "jobs" / f"{model}_fold{f}"
        jd.mkdir(parents=True, exist_ok=True)
        (jd / "FAILED.txt").write_text(traceback.format_exc())
        return dict(model=model, partition=p, fold=f, status=f"FAILED: {e}")


def _pool(devs):
    ctx = mp.get_context("spawn")
    slots = ctx.Manager().Queue()
    for d in devs:
        slots.put(d)
    return ctx.Pool(len(devs), initializer=_init, initargs=(slots, _env()))


# ============================================================================ notebook steps
def run_meta(notebook, p, extra=None):
    import sklearn
    import torch
    meta = dict(notebook=notebook, partition=p, seed=F.seed_of(p), python=platform.python_version(),
                numpy=np.__version__, sklearn=sklearn.__version__, pandas=pd.__version__, torch=torch.__version__,
                cpu_count=os.cpu_count(), gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                epochs=F.EPOCHS, started=time.strftime("%Y-%m-%d %H:%M:%S %Z"), core_sha=F.sha16(F.__file__),
                run_sha=F.sha16(__file__), partitions_sha=F.sha16(F.PARTITIONS),
                model_cmp_core_sha=F.sha16(Path(F.C.__file__)))
    try:
        meta["repo_commit"] = subprocess.check_output(["git", "-C", str(F.REPO), "rev-parse", "HEAD"]).decode().strip()
    except Exception as e:                                               # noqa: BLE001
        meta["repo_commit"] = f"unknown ({e})"
    meta.update(extra or {})
    _json(OUT / f"p{p}" / "run_meta.json", meta)
    print(json.dumps(meta, indent=1))
    return meta


def setup(p, input_root="/kaggle/input"):
    os.environ["F1_INPUT"] = input_root
    F.C.locate_all(input_root)
    gate(F.C.gate_D0(), p)


def gates(p):
    gate(F.gate_const(), p)
    gate(F.gate_vec(), p)
    gate(F.gate_mlp(), p)
    g = F.gate_GG0()
    PROJ["score_s_per_fold"] = float(np.mean(g["fold_seconds"]))
    gate(g, p)
    F.C._CACHE.clear()


def smoke(p):
    """one tiny end-to-end job per model (2 epochs, 2000 windows, one held-out case), then W concurrent timing
    workers -> projected wall time of the full run (LPT schedule over W workers)."""
    import torch
    t0 = time.time()
    for model in F.MODELS:
        F.job(p, model, 1, torch.device("cpu"), OUT, smoke=True)
    devs, mode = devices()
    with _pool(devs) as pool:
        tm = pool.map(_timing_worker, range(len(devs)))
    ms = {m: float(np.median([t[m] for t in tm])) for m in F.MODELS}
    jobs = []
    for model in F.MODELS:
        for f, man in F.manifest(p).items():
            n = sum(F.clean_counts()[s] for s in man["train_cases"])
            steps = F.EPOCHS * ((n + F.BATCH - 1) // F.BATCH)
            sc = PROJ.get("score_s_per_fold", 120) * 3          # single-thread scoring, conservative x3
            jobs.append(dict(model=model, fold=f, n_train=n, proj_h=(ms[model] * steps / 1000 + sc) / 3600 * 1.10))
    jobs.sort(key=lambda j: -j["proj_h"])
    load = [0.0] * len(devs)
    for j in jobs:                                              # LPT: longest job to the least-loaded worker
        i = int(np.argmin(load)); load[i] += j["proj_h"]
    PROJ.update(mode=mode, workers=len(devs), devices=devs, ms_per_step=ms, timing_workers=tm, jobs=jobs,
                projected_wall_h=max(load), smoke_s=round(time.time() - t0), elapsed_h=hours(),
                fits=hours() + max(load) <= REFUSE_H)
    _json(OUT / f"p{p}" / "projection.json", PROJ)
    F.log(f"SMOKE: mode {mode}, {len(devs)} workers, ms/step under concurrency {ms}; projected wall time "
          f"{max(load):.2f} h (+ {hours():.2f} h elapsed) -> {'FITS' if PROJ['fits'] else 'DOES NOT FIT'}")
    return PROJ


def import_previous(p, input_root="/kaggle/input"):
    """reuse finished jobs of a previous version of this notebook attached as input (same code and partitions)."""
    core, parts_sha = F.sha16(F.__file__), F.sha16(F.PARTITIONS)
    got = []
    for d in Path(input_root).rglob(f"f1_repcv/p{p}/jobs/*/DONE.json"):
        done = json.loads(d.read_text())
        if done.get("core_sha") != core or done.get("partitions_sha") != parts_sha:
            F.log(f"not reused (different code or partitions): {d.parent}")
            continue
        dst = OUT / f"p{p}" / "jobs" / d.parent.name
        if not (dst / "DONE.json").exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(d.parent, dst)
            got.append(d.parent.name)
    F.log(f"reused finished jobs from a previous version: {got or 'none'}")
    return got


def full(p, refuse_if_not_fit=True):
    if refuse_if_not_fit and not PROJ.get("fits", False):
        raise SystemExit(f"projected {PROJ.get('projected_wall_h')} h does not fit the session ({REFUSE_H} h); "
                         "jobs that fit can be run with full(p, refuse_if_not_fit=False)")
    done = {d.parent.name for d in (OUT / f"p{p}" / "jobs").glob("*/DONE.json")}
    todo = [j for j in PROJ["jobs"] if f"{j['model']}_fold{j['fold']}" not in done]
    devs, _ = devices()
    res = [dict(model=m.split("_fold")[0], partition=p, fold=int(m.split("_fold")[1]), status="reused") for m in done]
    if todo:
        with _pool(devs) as pool:
            for r in pool.imap_unordered(_job_worker, [(p, j["model"], j["fold"], j["proj_h"]) for j in todo], 1):
                res.append(r); _json(OUT / f"p{p}" / "jobs.json", res)
                F.log(f"job finished: {r}")
    _json(OUT / f"p{p}" / "jobs.json", res)
    failed = [r for r in res if r["status"] not in ("done", "reused")]
    if failed:
        F.log(f"jobs not done: {failed}")
    return res


def merge(p):
    jd = OUT / f"p{p}" / "jobs"
    for name in ("per_subject", "curves", "auroc", "seizures", "training_log"):
        parts = [pd.read_csv(d.parent / f"{name}.csv") for d in sorted(jd.glob("*/DONE.json"))]
        if parts:
            pd.concat(parts, ignore_index=True).to_csv(OUT / f"p{p}" / f"{name}.csv", index=False)
    n_done = len(list(jd.glob("*/DONE.json")))
    F.log(f"merged {n_done} of 8 jobs of partition {p}")
    return n_done


def summary(p):
    f = OUT / f"p{p}" / "per_subject.csv"
    if not f.exists():
        return
    d = pd.read_csv(f)
    d = d[(d.point == "own") & (d.stage == "localized")]
    s = d.assign(sens=d.tp / d.n_ref, fp_day=d.fp / d.nonict_h * 24,
                 f1=2 * d.tp / (2 * d.tp + d.fp + (d.n_ref - d.tp))).groupby("model")[["sens", "fp_day", "f1"]].mean()
    s["n_cases"] = d.groupby("model").size()
    print("\nUNVERIFIED per-case means (own point, after CPD) -- not a result before verification:")
    print(s.round(3).to_string())

In [ ]:
%%writefile {SRC}/partitions.json
{
 "plan": "docs/FUTURE_RESEARCH_PLAN.md rev 1, section 3",
 "rule": "stratified group 4-fold, unit = subject (chb01+chb21 one group); groups sorted by seizure count (results/v9_x3/folds.json) descending, ties by a random key; consecutive blocks of 4 groups; each block dealt to the folds by a random permutation; numpy.random.default_rng(r) for partition r",
 "folds_json_sha256": "f945b4f236f161007deff1acf1f054271d8edb78398b0be4a9cd9f3d42bbd6cf",
 "seizure_counts_by_group": {
  "chb12": 27,
  "chb15": 20,
  "chb24": 16,
  "chb13": 12,
  "chb01": 11,
  "chb06": 10,
  "chb16": 10,
  "chb14": 8,
  "chb20": 8,
  "chb03": 7,
  "chb10": 7,
  "chb23": 7,
  "chb18": 6,
  "chb05": 5,
  "chb08": 5,
  "chb04": 4,
  "chb09": 4,
  "chb02": 3,
  "chb07": 3,
  "chb11": 3,
  "chb17": 3,
  "chb19": 3,
  "chb22": 3
 },
 "cases_by_group": {
  "chb12": [
   "chb12"
  ],
  "chb15": [
   "chb15"
  ],
  "chb24": [
   "chb24"
  ],
  "chb13": [
   "chb13"
  ],
  "chb01": [
   "chb01",
   "chb21"
  ],
  "chb06": [
   "chb06"
  ],
  "chb16": [
   "chb16"
  ],
  "chb14": [
   "chb14"
  ],
  "chb20": [
   "chb20"
  ],
  "chb03": [
   "chb03"
  ],
  "chb10": [
   "chb10"
  ],
  "chb23": [
   "chb23"
  ],
  "chb18": [
   "chb18"
  ],
  "chb05": [
   "chb05"
  ],
  "chb08": [
   "chb08"
  ],
  "chb04": [
   "chb04"
  ],
  "chb09": [
   "chb09"
  ],
  "chb02": [
   "chb02"
  ],
  "chb07": [
   "chb07"
  ],
  "chb11": [
   "chb11"
  ],
  "chb17": [
   "chb17"
  ],
  "chb19": [
   "chb19"
  ],
  "chb22": [
   "chb22"
  ]
 },
 "pairwise_ari_random": {
  "mean": 0.032,
  "max": 0.2101
 },
 "git_commit_at_write": "1e653fa233344f262379bfc11b8c8b172fe0d562",
 "partitions": {
  "0": {
   "seed": 42,
   "rule": "committed CV (results/v9_x3/folds.json); not regenerated",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb12",
      "chb14",
      "chb20",
      "chb04",
      "chb09"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb10",
      "chb11",
      "chb13",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb14",
      "chb20",
      "chb04",
      "chb09"
     ],
     "seizures": 51
    },
    "2": {
     "held_out_cases": [
      "chb15",
      "chb16",
      "chb03",
      "chb08",
      "chb02",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb16",
      "chb03",
      "chb08",
      "chb02",
      "chb22"
     ],
     "seizures": 48
    },
    "3": {
     "held_out_cases": [
      "chb24",
      "chb06",
      "chb10",
      "chb05",
      "chb07",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb08",
      "chb09",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb06",
      "chb10",
      "chb05",
      "chb07",
      "chb19"
     ],
     "seizures": 44
    },
    "4": {
     "held_out_cases": [
      "chb13",
      "chb01",
      "chb21",
      "chb23",
      "chb18",
      "chb11",
      "chb17"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb19",
      "chb20",
      "chb22",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb01",
      "chb23",
      "chb18",
      "chb11",
      "chb17"
     ],
     "seizures": 42
    }
   },
   "ari_vs_p0": 1.0
  },
  "1": {
   "seed": 1,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb24",
      "chb16",
      "chb10",
      "chb18",
      "chb07"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb08",
      "chb09",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb17",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb16",
      "chb10",
      "chb18",
      "chb07"
     ],
     "seizures": 42
    },
    "2": {
     "held_out_cases": [
      "chb13",
      "chb01",
      "chb21",
      "chb23",
      "chb09",
      "chb11",
      "chb22"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb10",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb01",
      "chb23",
      "chb09",
      "chb11",
      "chb22"
     ],
     "seizures": 40
    },
    "3": {
     "held_out_cases": [
      "chb15",
      "chb14",
      "chb03",
      "chb05",
      "chb02",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb04",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb14",
      "chb03",
      "chb05",
      "chb02",
      "chb19"
     ],
     "seizures": 46
    },
    "4": {
     "held_out_cases": [
      "chb12",
      "chb06",
      "chb20",
      "chb08",
      "chb04",
      "chb17"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb05",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb06",
      "chb20",
      "chb08",
      "chb04",
      "chb17"
     ],
     "seizures": 57
    }
   },
   "ari_vs_p0": 0.0939
  },
  "2": {
   "seed": 2,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb24",
      "chb14",
      "chb23",
      "chb08",
      "chb17"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22"
     ],
     "held_out_groups": [
      "chb24",
      "chb14",
      "chb23",
      "chb08",
      "chb17"
     ],
     "seizures": 39
    },
    "2": {
     "held_out_cases": [
      "chb12",
      "chb01",
      "chb21",
      "chb10",
      "chb05",
      "chb02",
      "chb11"
     ],
     "train_cases": [
      "chb03",
      "chb04",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb01",
      "chb10",
      "chb05",
      "chb02",
      "chb11"
     ],
     "seizures": 56
    },
    "3": {
     "held_out_cases": [
      "chb13",
      "chb16",
      "chb20",
      "chb04",
      "chb09",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb19",
      "chb21",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb16",
      "chb20",
      "chb04",
      "chb09",
      "chb22"
     ],
     "seizures": 41
    },
    "4": {
     "held_out_cases": [
      "chb15",
      "chb06",
      "chb03",
      "chb18",
      "chb07",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb04",
      "chb05",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb16",
      "chb17",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb06",
      "chb03",
      "chb18",
      "chb07",
      "chb19"
     ],
     "seizures": 49
    }
   },
   "ari_vs_p0": -0.0222
  },
  "3": {
   "seed": 3,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb12",
      "chb01",
      "chb21",
      "chb20",
      "chb08",
      "chb04",
      "chb02"
     ],
     "train_cases": [
      "chb03",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb01",
      "chb20",
      "chb08",
      "chb04",
      "chb02"
     ],
     "seizures": 58
    },
    "2": {
     "held_out_cases": [
      "chb15",
      "chb14",
      "chb03",
      "chb18",
      "chb11"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb12",
      "chb13",
      "chb16",
      "chb17",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb14",
      "chb03",
      "chb18",
      "chb11"
     ],
     "seizures": 44
    },
    "3": {
     "held_out_cases": [
      "chb13",
      "chb06",
      "chb10",
      "chb05",
      "chb17",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb07",
      "chb08",
      "chb09",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb18",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb06",
      "chb10",
      "chb05",
      "chb17",
      "chb19"
     ],
     "seizures": 40
    },
    "4": {
     "held_out_cases": [
      "chb24",
      "chb16",
      "chb23",
      "chb09",
      "chb07",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb08",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21"
     ],
     "held_out_groups": [
      "chb24",
      "chb16",
      "chb23",
      "chb09",
      "chb07",
      "chb22"
     ],
     "seizures": 43
    }
   },
   "ari_vs_p0": 0.0707
  },
  "4": {
   "seed": 4,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb24",
      "chb16",
      "chb23",
      "chb18",
      "chb17",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb19",
      "chb20",
      "chb21"
     ],
     "held_out_groups": [
      "chb24",
      "chb16",
      "chb23",
      "chb18",
      "chb17",
      "chb22"
     ],
     "seizures": 45
    },
    "2": {
     "held_out_cases": [
      "chb12",
      "chb14",
      "chb03",
      "chb08",
      "chb11",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb13",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb14",
      "chb03",
      "chb08",
      "chb11",
      "chb19"
     ],
     "seizures": 53
    },
    "3": {
     "held_out_cases": [
      "chb15",
      "chb01",
      "chb21",
      "chb10",
      "chb05",
      "chb07"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb06",
      "chb08",
      "chb09",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb01",
      "chb10",
      "chb05",
      "chb07"
     ],
     "seizures": 46
    },
    "4": {
     "held_out_cases": [
      "chb13",
      "chb06",
      "chb20",
      "chb04",
      "chb09",
      "chb02"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb05",
      "chb07",
      "chb08",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb06",
      "chb20",
      "chb04",
      "chb09",
      "chb02"
     ],
     "seizures": 41
    }
   },
   "ari_vs_p0": 0.001
  },
  "5": {
   "seed": 5,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb24",
      "chb14",
      "chb20",
      "chb09",
      "chb02",
      "chb17"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb14",
      "chb20",
      "chb09",
      "chb02",
      "chb17"
     ],
     "seizures": 42
    },
    "2": {
     "held_out_cases": [
      "chb15",
      "chb01",
      "chb21",
      "chb10",
      "chb18",
      "chb11"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb12",
      "chb13",
      "chb14",
      "chb16",
      "chb17",
      "chb19",
      "chb20",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb01",
      "chb10",
      "chb18",
      "chb11"
     ],
     "seizures": 47
    },
    "3": {
     "held_out_cases": [
      "chb12",
      "chb06",
      "chb03",
      "chb05",
      "chb04",
      "chb07"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb06",
      "chb03",
      "chb05",
      "chb04",
      "chb07"
     ],
     "seizures": 56
    },
    "4": {
     "held_out_cases": [
      "chb13",
      "chb16",
      "chb23",
      "chb08",
      "chb19",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb16",
      "chb23",
      "chb08",
      "chb19",
      "chb22"
     ],
     "seizures": 40
    }
   },
   "ari_vs_p0": 0.0475
  },
  "6": {
   "seed": 6,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb12",
      "chb01",
      "chb21",
      "chb20",
      "chb18",
      "chb04"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb19",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb01",
      "chb20",
      "chb18",
      "chb04"
     ],
     "seizures": 56
    },
    "2": {
     "held_out_cases": [
      "chb13",
      "chb14",
      "chb10",
      "chb05",
      "chb17",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb11",
      "chb12",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb14",
      "chb10",
      "chb05",
      "chb17",
      "chb22"
     ],
     "seizures": 38
    },
    "3": {
     "held_out_cases": [
      "chb24",
      "chb16",
      "chb23",
      "chb08",
      "chb07",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb22"
     ],
     "held_out_groups": [
      "chb24",
      "chb16",
      "chb23",
      "chb08",
      "chb07",
      "chb19"
     ],
     "seizures": 44
    },
    "4": {
     "held_out_cases": [
      "chb15",
      "chb06",
      "chb03",
      "chb09",
      "chb02",
      "chb11"
     ],
     "train_cases": [
      "chb01",
      "chb04",
      "chb05",
      "chb07",
      "chb08",
      "chb10",
      "chb12",
      "chb13",
      "chb14",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb06",
      "chb03",
      "chb09",
      "chb02",
      "chb11"
     ],
     "seizures": 47
    }
   },
   "ari_vs_p0": 0.0242
  },
  "7": {
   "seed": 7,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb12",
      "chb14",
      "chb20",
      "chb04",
      "chb07",
      "chb17"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb05",
      "chb06",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb14",
      "chb20",
      "chb04",
      "chb07",
      "chb17"
     ],
     "seizures": 53
    },
    "2": {
     "held_out_cases": [
      "chb15",
      "chb16",
      "chb03",
      "chb18",
      "chb11",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb12",
      "chb13",
      "chb14",
      "chb17",
      "chb19",
      "chb20",
      "chb21",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb16",
      "chb03",
      "chb18",
      "chb11",
      "chb22"
     ],
     "seizures": 49
    },
    "3": {
     "held_out_cases": [
      "chb24",
      "chb06",
      "chb10",
      "chb05",
      "chb09",
      "chb02"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb04",
      "chb07",
      "chb08",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb06",
      "chb10",
      "chb05",
      "chb09",
      "chb02"
     ],
     "seizures": 45
    },
    "4": {
     "held_out_cases": [
      "chb13",
      "chb01",
      "chb21",
      "chb23",
      "chb08",
      "chb19"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb22",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb01",
      "chb23",
      "chb08",
      "chb19"
     ],
     "seizures": 38
    }
   },
   "ari_vs_p0": 0.2333
  },
  "8": {
   "seed": 8,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb15",
      "chb01",
      "chb21",
      "chb10",
      "chb09",
      "chb07",
      "chb19"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb08",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb01",
      "chb10",
      "chb09",
      "chb07",
      "chb19"
     ],
     "seizures": 48
    },
    "2": {
     "held_out_cases": [
      "chb13",
      "chb20",
      "chb23",
      "chb18",
      "chb17",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb19",
      "chb21",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb20",
      "chb23",
      "chb18",
      "chb17",
      "chb22"
     ],
     "seizures": 39
    },
    "3": {
     "held_out_cases": [
      "chb12",
      "chb16",
      "chb03",
      "chb08",
      "chb04"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb16",
      "chb03",
      "chb08",
      "chb04"
     ],
     "seizures": 53
    },
    "4": {
     "held_out_cases": [
      "chb24",
      "chb06",
      "chb14",
      "chb05",
      "chb02",
      "chb11"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb04",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb12",
      "chb13",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb06",
      "chb14",
      "chb05",
      "chb02",
      "chb11"
     ],
     "seizures": 45
    }
   },
   "ari_vs_p0": 0.0939
  },
  "9": {
   "seed": 9,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb15",
      "chb16",
      "chb23",
      "chb05",
      "chb02",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb04",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb16",
      "chb23",
      "chb05",
      "chb02",
      "chb22"
     ],
     "seizures": 48
    },
    "2": {
     "held_out_cases": [
      "chb13",
      "chb20",
      "chb03",
      "chb18",
      "chb07"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb04",
      "chb05",
      "chb06",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb19",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb20",
      "chb03",
      "chb18",
      "chb07"
     ],
     "seizures": 36
    },
    "3": {
     "held_out_cases": [
      "chb24",
      "chb01",
      "chb21",
      "chb10",
      "chb09",
      "chb11",
      "chb17"
     ],
     "train_cases": [
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb08",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb18",
      "chb19",
      "chb20",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb01",
      "chb10",
      "chb09",
      "chb11",
      "chb17"
     ],
     "seizures": 44
    },
    "4": {
     "held_out_cases": [
      "chb12",
      "chb06",
      "chb14",
      "chb08",
      "chb04",
      "chb19"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb05",
      "chb07",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb06",
      "chb14",
      "chb08",
      "chb04",
      "chb19"
     ],
     "seizures": 57
    }
   },
   "ari_vs_p0": 0.0707
  },
  "10": {
   "seed": 10,
   "rule": "rank blocks of 4, numpy.random.default_rng(r)",
   "folds": {
    "1": {
     "held_out_cases": [
      "chb12",
      "chb20",
      "chb23",
      "chb05",
      "chb04"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb06",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb11",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb19",
      "chb21",
      "chb22",
      "chb24"
     ],
     "held_out_groups": [
      "chb12",
      "chb20",
      "chb23",
      "chb05",
      "chb04"
     ],
     "seizures": 51
    },
    "2": {
     "held_out_cases": [
      "chb15",
      "chb06",
      "chb14",
      "chb18",
      "chb11",
      "chb17"
     ],
     "train_cases": [
      "chb01",
      "chb02",
      "chb03",
      "chb04",
      "chb05",
      "chb07",
      "chb08",
      "chb09",
      "chb10",
      "chb12",
      "chb13",
      "chb16",
      "chb19",
      "chb20",
      "chb21",
      "chb22",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb15",
      "chb06",
      "chb14",
      "chb18",
      "chb11",
      "chb17"
     ],
     "seizures": 50
    },
    "3": {
     "held_out_cases": [
      "chb24",
      "chb01",
      "chb21",
      "chb03",
      "chb09",
      "chb07",
      "chb19"
     ],
     "train_cases": [
      "chb02",
      "chb04",
      "chb05",
      "chb06",
      "chb08",
      "chb10",
      "chb11",
      "chb12",
      "chb13",
      "chb14",
      "chb15",
      "chb16",
      "chb17",
      "chb18",
      "chb20",
      "chb22",
      "chb23"
     ],
     "held_out_groups": [
      "chb24",
      "chb01",
      "chb03",
      "chb09",
      "chb07",
      "chb19"
     ],
     "seizures": 44
    },
    "4": {
     "held_out_cases": [
      "chb13",
      "chb16",
      "chb10",
      "chb08",
      "chb02",
      "chb22"
     ],
     "train_cases": [
      "chb01",
      "chb03",
      "chb04",
      "chb05",
      "chb06",
      "chb07",
      "chb09",
      "chb11",
      "chb12",
      "chb14",
      "chb15",
      "chb17",
      "chb18",
      "chb19",
      "chb20",
      "chb21",
      "chb23",
      "chb24"
     ],
     "held_out_groups": [
      "chb13",
      "chb16",
      "chb10",
      "chb08",
      "chb02",
      "chb22"
     ],
     "seizures": 40
    }
   },
   "ari_vs_p0": 0.0707
  }
 }
}

In [ ]:
# START: sha check of the embedded files, imports, run metadata, data gate D0 (hard stop on wrong data)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
for fn, want in (("f1_core.py", "bf4bfd69d7669226"), ("f1_run.py", "8768801adf6b6cfb"), ("partitions.json", "fbf73129105e0fea")):
    assert sha(f"{SRC}/{fn}") == want, f"{fn}: sha {sha(f'{SRC}/{fn}')} != {want} (cell edited?)"
os.environ.update(THESIS_REPO=REPO, F1_PARTITIONS=SRC + "/partitions.json", F1_OUT=WORK + "/f1_repcv", F1_INPUT=INPUT)
sys.path.insert(0, SRC)
import pandas as pd, torch
import f1_run as R
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), "| CPU cores:", os.cpu_count())
try:
    R.run_meta(NOTEBOOK, P)
    R.setup(P, INPUT)
except SystemExit as e:
    raise RuntimeError(f"STOPPED: {e}") from None

In [ ]:
# GATES (CPU, about 10 min): G-const, G-vec (training loop = PyG loop), G-mlp (= model comparison loop), GG0 (chain on
# partition 0 with the committed GAE checkpoints = results/v9_x3 exactly)
try:
    R.gates(P)
except SystemExit as e:
    raise RuntimeError(f"STOPPED: {e}") from None
print("GATES PASS")

In [ ]:
# SMOKE (always): one 2-epoch end-to-end job per model + speed of the parallel workers -> projection.json
try:
    proj = R.smoke(P)
except SystemExit as e:
    raise RuntimeError(f"STOPPED: {e}") from None
print(json.dumps({k: proj[k] for k in ("mode", "workers", "devices", "ms_per_step", "projected_wall_h", "fits")}, indent=1))

In [ ]:
# FULL RUN: 8 jobs (GAE + MLP-AE x 4 folds) on the parallel workers; each job written as soon as it ends
t_full = time.time()
try:
    R.import_previous(P, INPUT)
    R.full(P)
except SystemExit as e:
    raise RuntimeError(f"STOPPED: {e}") from None
print(f"FULL done in {(time.time()-t_full)/3600:.2f} h")

In [ ]:
# END: merge, gates, jobs, unverified summary, measured runtime
n_done = R.merge(P)
print(json.dumps(R.GATES, indent=1, default=str))
jf = R.OUT / f"p{P}" / "jobs.json"
print(jf.read_text() if jf.exists() else "(no full run)")
R.summary(P)
mf = R.OUT / f"p{P}" / "run_meta.json"
meta = json.loads(mf.read_text())
meta.update(notebook_runtime_h=round((time.time() - T_NB) / 3600, 3), finished=time.strftime("%Y-%m-%d %H:%M:%S %Z"),
            jobs_done=n_done)
mf.write_text(json.dumps(meta, indent=1))
print(f"MEASURED notebook runtime: {meta['notebook_runtime_h']} h; jobs done: {n_done} of 8")
print(sorted(os.listdir(R.OUT / f"p{P}")))